# Neural Network Models for Steam Sales Prediction

This notebook loads the Steam games dataset, preprocesses it using the project pipeline, and trains five different neural network architectures on `log_copies_sold`.

**Architectures compared:**
| Model | Hidden Layers | Notes |
|-------|--------------|-------|
| NN1_Shallow | 256 → 128 | 2 layers, fast baseline |
| NN2_Medium | 512 → 256 → 128 | 3 layers |
| NN3_Deep | 512 → 256 → 128 → 64 | 4 layers |
| NN4_Wide | 1024 → 512 → 256 | 3 wide layers |
| NN5_DeepDrop | 512 → 256 → 128 → 64 → 32 | 5 layers + Dropout + BatchNorm |

All models use **ReLU** activations and are trained with **Adam** optimizer on **MSE loss** against the log-transformed target.

In [16]:
import sys
import logging
import warnings
warnings.filterwarnings('ignore')

# Make sure the project root is on the path
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Configure logging so training progress is visible
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)

print('Libraries imported successfully.')

Libraries imported successfully.


## 1. Load & Preprocess Data

In [17]:
from configs.config import RAW_MERGED_PATH
from src.data.loader import load_merged, validate_merged
from src.data.preprocessor import run_preprocessing_pipeline
from src.features.engineer import prepare_features

# Step 1 — Load raw merged CSV
print('Loading raw data...')
df_raw = load_merged(RAW_MERGED_PATH, verbose=True)
validate_merged(df_raw)
print(f'Raw shape: {df_raw.shape}')

16:50:44  INFO      Loading /Users/arjianma/CIS5450-FINAL-PROJECT/Data/games_merged.csv …


Loading raw data...

  games_merged.csv
  Rows: 115,289   Columns: 47

  Columns with >10% nulls:
    Movies                              100.0%
    Score rank                          100.0%
    Metacritic url                      96.4%
    Reviews                             89.6%
    Notes                               79.9%
    Website                             57.6%
    Support url                         53.5%
    Tags                                29.8%
    Support email                       12.0%



16:50:53  WARNING   copiesSold has 98 null values — check the merge.
16:50:53  INFO      Validation passed: 115289 rows × 47 cols


Raw shape: (115289, 47)


In [27]:
# Step 2 — Run 14-step preprocessing pipeline
print('Running preprocessing pipeline (post-release model)...')
df = run_preprocessing_pipeline(df_raw, post_release=False, verbose=True)
print(f'Processed shape: {df.shape}')

16:53:23  INFO        → drop useless columns
16:53:23  INFO        → drop target nulls


Running preprocessing pipeline (post-release model)...


16:53:23  INFO      Dropped 98 rows with null copiesSold.
16:53:23  INFO        → parse release date
16:53:24  INFO        → parse supported languages
16:53:26  INFO        → parse estimated owners
16:53:26  INFO        → parse genres
16:53:27  INFO        → parse categories
16:53:28  INFO        → parse tags
16:53:29  INFO      Top 30 tags (by frequency): ['Singleplayer', 'Indie', 'Action', 'Casual', 'Adventure', '2D', '3D', 'Strategy', 'Simulation', 'Puzzle']
16:53:30  INFO        → derive numeric features
16:53:30  INFO        → handle metacritic sparsity
16:53:31  INFO        → encode publisher class
16:53:31  INFO        → frequency-encode dev/pub
16:53:31  INFO        → log-transform skewed cols
16:53:31  INFO        → create log target
16:53:31  INFO        → drop post-release features
16:53:31  INFO      Launch-time mode: dropped 14 post-release columns.
16:53:31  INFO      Preprocessing complete: 115191 rows × 83 columns


Processed shape: (115191, 83)


In [19]:
# Step 3 — Build feature matrix and train/val/test splits
print('Building feature matrix and splitting data...')
data = prepare_features(df, post_release=True, return_pipeline=True)

X_train, X_val, X_test = data['X_train'], data['X_val'], data['X_test']
y_train, y_val, y_test = data['y_train'], data['y_val'], data['y_test']

print(f'\nData splits:')
print(f'  Train : {X_train.shape[0]:>7,} samples  x  {X_train.shape[1]} features')
print(f'  Val   : {X_val.shape[0]:>7,} samples')
print(f'  Test  : {X_test.shape[0]:>7,} samples')
print(f'\nTarget (log_copies_sold):')
print(f'  Train mean: {y_train.mean():.3f}  std: {y_train.std():.3f}')

16:51:04  INFO      Feature matrix: 93 columns (post-release mode)
16:51:04  INFO      Split sizes — train: 80633  val: 17279  test: 17279


Building feature matrix and splitting data...


16:51:04  INFO      Pipeline columns — continuous: 27  binary: 65  ordinal: 1



Data splits:
  Train :  80,633 samples  x  93 features
  Val   :  17,279 samples
  Test  :  17,279 samples

Target (log_copies_sold):
  Train mean: 6.411  std: 3.020


## 2. Check PyTorch & GPU Availability

In [24]:
try:
    import torch
    print(f'PyTorch version : {torch.__version__}')
    print(f'CUDA available  : {torch.cuda.is_available()}')
    if hasattr(torch.backends, "mps"):
        print(f'MPS available   : {torch.backends.mps.is_available()}')
    dev = (
        'cuda' if torch.cuda.is_available()
        else ('mps' if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
              else 'cpu')
    )
    print(f'\nUsing device    : {dev}')
except ImportError:
    print('PyTorch is NOT installed.  Install it with:')
    print('  pip install torch')
    print('\nFalling back — models will not train.')

PyTorch version : 2.8.0
CUDA available  : False
MPS available   : True

Using device    : mps


## 3. Architecture Overview

Inspect each model's layer structure and parameter count before training.

In [25]:
from src.models.nn import build_all_nn_models, NN_CONFIGS

input_dim = X_train.shape[1]
print(f'Input dimension: {input_dim} features\n')
print(f'{"Model":<20} {"Architecture":<35} {"Dropout":>8} {"BatchNorm":>10}')
print('-' * 80)
for key, cfg in NN_CONFIGS.items():
    arch = ' → '.join(str(h) for h in cfg['hidden_layers']) + ' → 1'
    print(f'{cfg["name"]:<20} {arch:<35} {cfg["dropout_rate"]:>8.1f} {str(cfg["batch_norm"]):>10}')

Input dimension: 93 features

Model                Architecture                         Dropout  BatchNorm
--------------------------------------------------------------------------------
NN1_Shallow          256 → 128 → 1                            0.0      False
NN2_Medium           512 → 256 → 128 → 1                      0.0      False
NN3_Deep             512 → 256 → 128 → 64 → 1                 0.0      False
NN4_Wide             1024 → 512 → 256 → 1                     0.0      False
NN5_DeepDrop         512 → 256 → 128 → 64 → 32 → 1            0.3       True


## 4. Train All Five Neural Network Models

In [26]:
from src.models.nn import NeuralNetModel, NN_CONFIGS, run_nn_models

# Train all five models and collect metrics
summary_df = run_nn_models(data, verbose=True)
print('\n', summary_df.to_string())

16:51:55  INFO      ============================================================
16:51:55  INFO      Training NN1_Shallow: PyTorch not installed.
16:51:55  INFO      ============================================================


ImportError: Install PyTorch before calling .fit().

## 5. Results Comparison Table

In [ ]:
display_cols = ['val_RMSE_log', 'val_MAE_log', 'val_R2_log',
                'test_RMSE_log', 'train_RMSE_log', 'dropout']
display_df = summary_df[display_cols].sort_values('val_RMSE_log')

print('\nNeural Network Model Comparison (sorted by val RMSE_log ↑ best):')
print('=' * 75)
display(display_df.style
    .background_gradient(cmap='RdYlGn_r', subset=['val_RMSE_log', 'test_RMSE_log'])
    .background_gradient(cmap='RdYlGn',   subset=['val_R2_log'])
    .format({
        'val_RMSE_log':   '{:.4f}',
        'val_MAE_log':    '{:.4f}',
        'val_R2_log':     '{:.4f}',
        'test_RMSE_log':  '{:.4f}',
        'train_RMSE_log': '{:.4f}',
        'dropout':        '{:.1f}',
    })
)

## 6. Training Loss Curves

Visualise how training and validation loss evolved for each architecture.

In [ ]:
from src.models.nn import NeuralNetModel, NN_CONFIGS

# Re-train individual models to capture history objects
# (run_nn_models above already trained them — re-run here for plotting)
trained_models = []
for key, cfg in NN_CONFIGS.items():
    m = NeuralNetModel(**cfg)
    m.fit(X_train, y_train, X_val=X_val, y_val=y_val, verbose=False)
    trained_models.append(m)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes_flat = axes.flatten()

colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']

for i, model in enumerate(trained_models):
    ax = axes_flat[i]
    train_loss = model.history.get('train_loss', [])
    val_loss   = model.history.get('val_loss',   [])

    ax.plot(train_loss, label='Train loss', color=colors[i], linewidth=2)
    if val_loss:
        ax.plot(val_loss, label='Val loss', color=colors[i], linestyle='--',
                linewidth=2, alpha=0.7)
    ax.set_title(model.name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

# Hide the unused 6th subplot
axes_flat[-1].set_visible(False)

fig.suptitle('Training & Validation Loss Curves — All NN Architectures',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Val RMSE Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

sorted_df = summary_df.sort_values('val_RMSE_log')
bar_colors = ['#4CAF50' if i == 0 else '#2196F3' for i in range(len(sorted_df))]

bars = ax.barh(sorted_df.index, sorted_df['val_RMSE_log'], color=bar_colors,
               edgecolor='white', height=0.6)

for bar, val in zip(bars, sorted_df['val_RMSE_log']):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=10)

ax.set_xlabel('Validation RMSE (log scale)', fontsize=12)
ax.set_title('Neural Network Models — Validation RMSE Comparison\n(lower = better)',
             fontsize=13, fontweight='bold')
ax.set_xlim(0, sorted_df['val_RMSE_log'].max() * 1.15)
ax.grid(axis='x', alpha=0.3)
ax.invert_yaxis()

plt.tight_layout()
plt.show()

## 8. Predicted vs Actual — Best Model

In [ ]:
import numpy as np

# Pick the best model by val RMSE
best_name = summary_df['val_RMSE_log'].idxmin()
best_model = next(m for m in trained_models if m.name == best_name)
print(f'Best model: {best_name}')

y_pred_log = best_model.predict(X_test)
y_true_log = np.asarray(y_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Scatter: predicted vs actual (log scale) ---
ax = axes[0]
ax.scatter(y_true_log, y_pred_log, alpha=0.15, s=5, color='steelblue')
lims = [min(y_true_log.min(), y_pred_log.min()),
        max(y_true_log.max(), y_pred_log.max())]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
ax.set_xlabel('Actual log(copies sold + 1)', fontsize=11)
ax.set_ylabel('Predicted log(copies sold + 1)', fontsize=11)
ax.set_title(f'{best_name} — Predicted vs Actual (Test Set)', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

# --- Histogram of residuals ---
ax2 = axes[1]
residuals = y_pred_log - y_true_log
ax2.hist(residuals, bins=60, color='steelblue', edgecolor='white', alpha=0.85)
ax2.axvline(0, color='red', linestyle='--', linewidth=1.5)
ax2.set_xlabel('Residual (predicted − actual)', fontsize=11)
ax2.set_ylabel('Count', fontsize=11)
ax2.set_title(f'{best_name} — Residual Distribution', fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nTest metrics for {best_name}:')
for k, v in best_model.test_metrics.items():
    print(f'  {k:<15} {v:>12,.4f}')

## Summary

All five neural network architectures have been trained and evaluated. Key takeaways:

- **Training target**: `log1p(copiesSold)` — log-transforming the highly skewed target stabilises training.
- **All models use ReLU** activations throughout the hidden layers.
- **NN5_DeepDrop** uses an additional Dropout (p=0.3) + BatchNorm regularisation strategy to reduce overfitting through 5 layers.
- **Early stopping** (patience=10–20 epochs) prevents overfitting and reduces training time.
- Compare `val_RMSE_log` across models — lower is better. Compare with tree-based baselines in `src/models/advanced.py` for context.